# AG_PRAXIS NB11 — Pilot: Loading Windows With Their Capture Identity

Each attack class in this dataset was recorded in its own session, and eight of the
nineteen classes were recorded more than once. That matters because it is the only place
in the corpus where which session a window came from and which class it belongs to are
not the same fact. On a class recorded once, knowing the session tells you the class and
knowing the class tells you the session, so nothing can be measured about one while
holding the other still. On a class recorded several times there is a real question:
given a window of this class, which of its sessions did it come from?

This notebook is the first step of an experiment that needs that structure. Before
anything is trained I want to see the ground it stands on, so all this does is read the
saved windows together with the recording each one was cut from, work the recording names
back to session identifiers, and count what is actually there.

Nothing is trained here and nothing is written to disk. It prints the shapes of the
arrays, how many sessions the training partition holds, and how many training windows
belong to the eight classes recorded more than once.

The usual first cell: Drive, the repository, and the commit this ran at.

In [3]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

colab     : False
repo root : /Users/gina/Documents/AG_PRAXIS
git sha   : 8b78842 on main   WORKING TREE DIRTY
run date  : 2026-08-16


Configuration and inputs. The window arrays live on Drive because they are too large
for the repository, and the manifest that records how they were cut is committed, so the
two are checked against each other rather than either being trusted alone. Nothing is
written, so no output directory is created.

In [5]:
import json

import numpy as np
import pandas as pd

from src import captures as cap
from src import interventions as iv
from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"],
    "the manifest recording how the windows were cut",
)
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the saved sequence arrays")

MANIFEST = json.loads(MANIFEST_PATH.read_text())

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
SEQUENCES = {
    part: dict(MANIFEST["arrays"][f"sequences_{part}"]["by_class"])
    for part in ("train", "val", "test")
}

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 240)

print(f"manifest : {MANIFEST_PATH}")
print(f"arrays   : {ARRAY_DIR}")
print(f"seed     : {SEED}")
print(f"window, stride : {WINDOW}, {STRIDE}")
print(f"features : {len(FEATURES)}")
print(f"classes  : {len(CLASSES)}")
print("nothing is written by this notebook")

FileNotFoundError: the saved sequence arrays not found. Looked in: ['/content/drive/MyDrive/AG_PRAXIS_artifacts/NB04']

Reading the windows. The array file carries more than the windows and their labels: it
also records, for every window, which recording it was cut from. That column is what this
notebook is for, and it is the one thing the training runs so far have not read.

The three checks before anything is returned are the same ones any run makes. The columns
have to be the ones the manifest lists, the classes have to be the ones it lists, and the
file has to have been cut at the window and stride this configuration names. A file that
fails any of them describes a different corpus, and every count below it would be a count
of something else.

In [ ]:
def read_partition(name):
    path = ARRAY_DIR / f"sequences_{name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. The preprocessing step writes it.")
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X = npz["X"]
        y = npz["y"].astype("int64")
        recording_codes = npz["recording"].astype("int64")
        recordings = [str(v) for v in npz["recordings"]]
    print(f"  {path.name:<24} {str(X.shape):>22}   {X.dtype}")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    assert len(recording_codes) == len(y) == len(X)
    assert np.isfinite(X).all(), f"{path.name} holds a value that is not finite"
    return {"X": X, "y": y, "recording_codes": recording_codes, "recordings": recordings}


print("reading the sequence arrays")
TRAIN = read_partition("train")
TEST = read_partition("test")

print()
print(f"train windows : {len(TRAIN['y']):,}")
print(f"test windows  : {len(TEST['y']):,}")
print(f"each window   : {WINDOW} records of {len(FEATURES)} features, already scaled")
print(f"recordings named in the training file : {len(TRAIN['recordings'])}")
print(f"recordings named in the test file     : {len(TEST['recordings'])}")

for part, held in (("train", TRAIN), ("test", TEST)):
    counts = {CLASSES[c]: int(n) for c, n in enumerate(np.bincount(held["y"], minlength=len(CLASSES)))}
    assert counts == SEQUENCES[part], (
        f"the {part} array holds different per-class counts than the manifest records"
    )
print()
print("per-class counts in both partitions agree with the manifest")

From recording to session. A recording name carries the partition it belongs to and,
for a class recorded more than once, a chunk number. The session identifier is what is
left when those are stripped, and the project already has one rule for doing that, so it
is used here rather than a second rule being written that could disagree with it.

Every window then carries an integer code for its session, and those codes are what a
later experiment would group by.

In [ ]:
def capture_of(recording_name):
    """The capture a recording belongs to, by the project's own naming rule."""
    return cap.parse_capture(f"{recording_name}.pcap.csv")["capture_id"]


CAPTURE_OF_RECORDING = {name: capture_of(name) for name in TRAIN["recordings"]}
TRAIN_CAPTURES = [CAPTURE_OF_RECORDING[TRAIN["recordings"][c]] for c in TRAIN["recording_codes"]]

GROUP_CODES, GROUP_NAMES = iv.group_codes(TRAIN_CAPTURES)
GROUP_SIZES = iv.group_sizes(GROUP_CODES, len(GROUP_NAMES))
N_GROUPS = len(GROUP_NAMES)

CLASS_OF_GROUP = {}
for code, label_code in zip(GROUP_CODES, TRAIN["y"]):
    CLASS_OF_GROUP.setdefault(GROUP_NAMES[code], CLASSES[label_code])

CAPTURES_PER_CLASS = {}
for group, label in CLASS_OF_GROUP.items():
    CAPTURES_PER_CLASS.setdefault(label, []).append(group)

MULTI = {k: sorted(v) for k, v in CAPTURES_PER_CLASS.items() if len(v) > 1}
SINGLE = {k: sorted(v) for k, v in CAPTURES_PER_CLASS.items() if len(v) == 1}

# A capture belongs to exactly one class, so a capture that reached two would mean the
# naming rule had merged two recordings that are not the same session.
for group in GROUP_NAMES:
    labels = {CLASSES[c] for c, g in zip(TRAIN["y"], GROUP_CODES) if g == GROUP_NAMES.index(group)}
    assert len(labels) == 1, f"capture {group} carries more than one class: {sorted(labels)}"

print(f"captures over the training windows : {N_GROUPS}")
print(f"classes recorded more than once    : {len(MULTI)}")
print(f"classes recorded once              : {len(SINGLE)}")
print()
print("the classes recorded more than once, and their captures")
for label in sorted(MULTI):
    windows = sum(int(GROUP_SIZES[GROUP_NAMES.index(g)]) for g in MULTI[label])
    print(f"  {label:<12} {len(MULTI[label])} captures   {windows:>7,} training windows   "
          f"{', '.join(MULTI[label])}")

What the experiment would have to work with. The windows of the classes recorded more
than once are the only ones where a session can be asked about while the class is held
still, so their count is the size of the ground available, and everything else is
background.

In [ ]:
MULTI_CLASSES = sorted(MULTI)
MULTI_CAPTURES = sorted({g for label in MULTI_CLASSES for g in MULTI[label]})
MULTI_MASK = np.isin(TRAIN["y"], [CLASSES.index(c) for c in MULTI_CLASSES])

N_MULTI_WINDOWS = int(MULTI_MASK.sum())
N_TRAIN_WINDOWS = int(len(TRAIN["y"]))

POOLED_CHANCE = 1.0 / len(MULTI_CAPTURES)
HELD_FIXED_CHANCE = float(np.mean([1.0 / len(MULTI[c]) for c in MULTI_CLASSES]))

SUMMARY = pd.DataFrame([
    {
        "class": label,
        "captures": len(MULTI[label]),
        "train windows": int(GROUP_SIZES[[GROUP_NAMES.index(g) for g in MULTI[label]]].sum()),
        "smallest capture": int(min(GROUP_SIZES[GROUP_NAMES.index(g)] for g in MULTI[label])),
        "largest capture": int(max(GROUP_SIZES[GROUP_NAMES.index(g)] for g in MULTI[label])),
    }
    for label in MULTI_CLASSES
])

print(SUMMARY.to_string(index=False))
print()
print(f"classes recorded more than once : {len(MULTI_CLASSES)}")
print(f"captures across them            : {len(MULTI_CAPTURES)} of {N_GROUPS}")
print(f"training windows in them        : {N_MULTI_WINDOWS:,} of {N_TRAIN_WINDOWS:,}"
      f"   ({100 * N_MULTI_WINDOWS / N_TRAIN_WINDOWS:.1f}%)")
print()
print("guessing rates for a session identified from one of these windows")
print(f"  pooled over all {len(MULTI_CAPTURES)} captures        : {POOLED_CHANCE:.4f}")
print(f"  with the attack class held fixed  : {HELD_FIXED_CHANCE:.4f}")
print()
print("The second is the rate that matters. Pooled over every capture, naming the capture "
      "would name")
print("the class as well, so a model doing it well would only be telling the classes apart "
      "under")
print("another name.")

assert MULTI_MASK.sum() > 0, "no window belongs to a class recorded more than once"
assert len(MULTI_CAPTURES) + len(SINGLE) == N_GROUPS, (
    "the captures of the multi-capture classes and the single-capture classes do not "
    "account for every capture"
)

Choosing what to hold back. The experiment this is a pilot for needs some sessions kept
out of whatever the model trains on, so that a question can be put later to data the
training never saw. One capture is held back from each of the eight classes, and it is the
smallest of that class's captures, so the least possible data is taken away from the eight
classes that are the only place the question can be put at all.

Which capture is smallest is read off the window counts rather than chosen. Where two
captures of a class hold the same number of windows the choice is made by a seeded draw
rather than by whichever name happens to sort first, so the same captures come out every
time this runs, and the table records whether the draw was needed.

Nothing is split here and nothing is trained. This names the captures and counts what
falls on each side of the line.

In [ ]:
HOLDOUT_SEED = SEED
PICKER = np.random.default_rng(HOLDOUT_SEED)


def smallest_capture(label):
    """The capture of this class holding the fewest training windows.

    Returns the capture, the sizes of all of that class's captures, and whether a tie
    had to be broken. A tie is broken by a draw from the seeded generator above rather
    than by name order, so the choice does not depend on how the captures were named.
    """
    sizes = {g: int(GROUP_SIZES[GROUP_NAMES.index(g)]) for g in MULTI[label]}
    fewest = min(sizes.values())
    tied = sorted(g for g, n in sizes.items() if n == fewest)
    if len(tied) == 1:
        return tied[0], sizes, False
    return str(PICKER.choice(tied)), sizes, True


HELD_OUT = {}
HOLDOUT_ROWS = []
for label in MULTI_CLASSES:
    chosen, capture_sizes, tie_broken = smallest_capture(label)
    HELD_OUT[label] = chosen
    HOLDOUT_ROWS.append({
        "class": label,
        "held-out capture": chosen,
        "its windows": capture_sizes[chosen],
        "windows remaining": sum(n for g, n in capture_sizes.items() if g != chosen),
        "captures remaining": len(capture_sizes) - 1,
        "tie broken": tie_broken,
    })

HOLDOUT = pd.DataFrame(HOLDOUT_ROWS)
HELD_OUT_CAPTURES = sorted(HELD_OUT.values())

HELD_OUT_MASK = np.isin(GROUP_CODES, [GROUP_NAMES.index(g) for g in HELD_OUT_CAPTURES])
DOMAIN_MASK = MULTI_MASK & ~HELD_OUT_MASK

CHANCE_AFTER = float(np.mean([1.0 / (len(MULTI[c]) - 1) for c in MULTI_CLASSES]))

print(HOLDOUT.to_string(index=False))
print()
print(f"captures held back          : {len(HELD_OUT_CAPTURES)} of {len(MULTI_CAPTURES)}")
print(f"captures left to train on   : {len(MULTI_CAPTURES) - len(HELD_OUT_CAPTURES)}")
print()
print(f"windows to the domain head  : {int(DOMAIN_MASK.sum()):>8,}")
print(f"windows held back to probe  : {int(HELD_OUT_MASK.sum()):>8,}")
print(f"                              {'-' * 8}")
print(f"windows in the eight classes: {N_MULTI_WINDOWS:>8,}")
print()
print("guessing rate for a session named with the attack class held fixed")
print(f"  over every capture of the eight classes : {HELD_FIXED_CHANCE:.4f}")
print(f"  over what is left after the holdout      : {CHANCE_AFTER:.4f}")

assert set(HELD_OUT) == set(MULTI_CLASSES), "a class did not get a held-out capture"
assert len(HELD_OUT_CAPTURES) == len(set(HELD_OUT_CAPTURES)) == len(MULTI_CLASSES), (
    "the held-out captures are not one distinct capture per class"
)
assert set(HELD_OUT_CAPTURES) <= set(MULTI_CAPTURES), (
    "a held-out capture is not one of the captures of the eight classes"
)
assert not bool((HELD_OUT_MASK & ~MULTI_MASK).any()), (
    "a held-out capture holds windows outside the eight classes"
)
assert not bool((HELD_OUT_MASK & DOMAIN_MASK).any()), "the two masks overlap"
assert int(HELD_OUT_MASK.sum()) + int(DOMAIN_MASK.sum()) == N_MULTI_WINDOWS, (
    "the two masks do not account for every window of the eight classes"
)
assert all(len(MULTI[c]) - 1 >= 2 for c in MULTI_CLASSES), (
    "a class is left with fewer than two captures, so within that class one capture "
    "could not be told from another in what the model would train on"
)

One last thing to look at before any of this is built on, because it decides where a
session can be measured at all. A session can only be told from another session of the
same class if that class has more than one session in the partition being measured. That
holds on the training partition and it is worth checking whether it holds on the others,
since a number measured only where the training objective already looked is a weaker
number than one measured on held-out data.

In [ ]:
def captures_per_class(held):
    """How many distinct captures each class has in one partition."""
    names = [capture_of(held["recordings"][c]) for c in held["recording_codes"]]
    per_class = {}
    for label_code, group in zip(held["y"], names):
        per_class.setdefault(CLASSES[label_code], set()).add(group)
    return {label: len(groups) for label, groups in sorted(per_class.items())}


TRAIN_PER_CLASS = captures_per_class(TRAIN)
TEST_PER_CLASS = captures_per_class(TEST)

PARTITIONS = pd.DataFrame([
    {
        "class": label,
        "captures in train": TRAIN_PER_CLASS.get(label, 0),
        "captures in test": TEST_PER_CLASS.get(label, 0),
    }
    for label in CLASSES
])

print(PARTITIONS.to_string(index=False))
print()
print(f"classes with more than one capture in train : "
      f"{sum(1 for n in TRAIN_PER_CLASS.values() if n > 1)}")
print(f"classes with more than one capture in test  : "
      f"{sum(1 for n in TEST_PER_CLASS.values() if n > 1)}")
print()
print("Where a class has one capture in a partition, asking which of its captures a window "
      "came")
print("from has one answer, so the question cannot be put on that partition at all.")

Splitting each capture the adversary trains on, so the probe has somewhere to stand.

The captures held back give an unseen session for the attack head, but they cannot answer
the session question: each class has exactly one of them, so asking which of this class's
captures a window came from has one answer there. The question can only be put where a
class has several sessions, which is the captures the adversary trains on, and asking it
of the very windows the adversary optimised over would measure memorisation. So each of
those captures is cut in two along its own timeline: the earlier part trains the
adversary, the later part is kept back to probe it.

The cut is by position in the recording and never at random, for the reason the project's
split protocol already gives: rows next to each other in a file can come from the same
burst of traffic, and a random cut would put related rows on both sides.

A window is fifty records and the stride is twenty-five, so neighbouring windows share
half their rows and a cut cannot simply fall between two of them. Window k covers rows
[k*stride, k*stride + window), so windows k and k+d share a row whenever d*stride is less
than window. The smallest separation sharing nothing is the window over the stride, rounded
up, and the number of windows thrown away at the boundary is one less than that. Here that
comes to one. It is computed rather than written down, because at another window or stride
it would not be one, and a boundary that leaks is a probe measuring what it trained on.

In [ ]:
PROBE_FRACTION = 0.30

# Windows k and k+d share a row whenever d * STRIDE < WINDOW, so the smallest separation
# that shares nothing is ceil(WINDOW / STRIDE) and the number to drop between the two
# sides is one less. At 50 and 25 that is one window; at 50 and 20 it would be two.
MIN_SEPARATION = int(np.ceil(WINDOW / STRIDE))
GAP = MIN_SEPARATION - 1

# start_row is the absolute position in the source file of a window's first record. The
# reader above does not take it, because nothing until now needed to know where inside a
# recording a window sits, so it is read here.
with np.load(ARRAY_DIR / "sequences_train.npz", allow_pickle=False) as npz:
    START_ROW = npz["start_row"].astype("int64")
assert len(START_ROW) == len(TRAIN["y"]), "start_row does not cover the training windows"

SPLIT_ROWS = []
ADVERSARY_PARTS, PROBE_PARTS, DROPPED_PARTS = [], [], []

for capture in DOMAIN_CAPTURES:
    where = np.flatnonzero(GROUP_CODES == GROUP_NAMES.index(capture))
    order = where[np.argsort(START_ROW[where], kind="mergesort")]

    n = len(order)
    n_adversary = int(np.floor(n * (1.0 - PROBE_FRACTION)))
    n_probe = n - n_adversary - GAP
    if n_adversary < 1 or n_probe < 1:
        raise ValueError(
            f"{capture} holds {n} windows, too few to cut {PROBE_FRACTION:.0%} off the end "
            f"and still drop {GAP} at the boundary"
        )

    adversary = order[:n_adversary]
    dropped = order[n_adversary : n_adversary + GAP]
    probe = order[n_adversary + GAP :]

    # Consecutive windows of a capture overlap, so the rows one side covers are a single
    # unbroken stretch from its first window's first row to its last window's last row.
    # Comparing the two stretches is therefore the whole check, not a sample of it.
    last_adversary_row = int(START_ROW[adversary].max() + WINDOW - 1)
    first_probe_row = int(START_ROW[probe].min())

    ADVERSARY_PARTS.append(adversary)
    PROBE_PARTS.append(probe)
    DROPPED_PARTS.append(dropped)
    SPLIT_ROWS.append({
        "capture": capture,
        "class": CLASS_OF_GROUP[capture],
        "windows": n,
        "adversary": len(adversary),
        "dropped": len(dropped),
        "probe": len(probe),
        "last adversary row": last_adversary_row,
        "first probe row": first_probe_row,
        "rows between": first_probe_row - last_adversary_row - 1,
        "in file order": bool(np.array_equal(order, where)),
    })

PROBE_SPLIT = pd.DataFrame(SPLIT_ROWS)
ADVERSARY_INDEX = np.sort(np.concatenate(ADVERSARY_PARTS))
PROBE_INDEX = np.sort(np.concatenate(PROBE_PARTS))
DROPPED_INDEX = (np.sort(np.concatenate(DROPPED_PARTS)) if GAP
                 else np.array([], dtype="int64"))

print(f"window {WINDOW}, stride {STRIDE}")
print(f"  windows this far apart share no record : {MIN_SEPARATION}")
print(f"  so windows dropped at each boundary    : {GAP}")
print(f"  probe share of each capture            : {PROBE_FRACTION:.0%}, taken from the end")
print()
print(PROBE_SPLIT.to_string(index=False))
print()
print(f"captures split          : {len(PROBE_SPLIT)}")
print(f"adversary windows       : {len(ADVERSARY_INDEX):>7,}")
print(f"probe windows           : {len(PROBE_INDEX):>7,}")
print(f"dropped at boundaries   : {len(DROPPED_INDEX):>7,}")
print(f"                          {'-' * 7}")
print(f"windows in those captures: {int(DOMAIN_MASK.sum()):>6,}")

LEAKING = PROBE_SPLIT[PROBE_SPLIT["last adversary row"] >= PROBE_SPLIT["first probe row"]]
assert LEAKING.empty, (
    "these captures have an adversary window sharing a record with a probe window: "
    f"{list(LEAKING['capture'])}"
)
assert bool(PROBE_SPLIT["in file order"].all()), (
    "a capture's windows are not in file order in the array, so ordering by start_row "
    "reordered them and the cut is not a cut along the recording"
)
assert bool((PROBE_SPLIT["rows between"] >= 0).all())
assert len(np.intersect1d(ADVERSARY_INDEX, PROBE_INDEX)) == 0
assert len(np.intersect1d(ADVERSARY_INDEX, DROPPED_INDEX)) == 0
assert len(np.intersect1d(PROBE_INDEX, DROPPED_INDEX)) == 0
assert (len(ADVERSARY_INDEX) + len(PROBE_INDEX) + len(DROPPED_INDEX)
        == int(DOMAIN_MASK.sum())), (
    "the three parts do not account for every window of the captures being split"
)
assert bool(DOMAIN_MASK[ADVERSARY_INDEX].all()) and bool(DOMAIN_MASK[PROBE_INDEX].all())
print()
print("no record appears on both sides of any capture's cut")

The model, built and not trained. It has the classifier this project already trains
and one addition: a second head reading the same representation, whose job is to name the
recording session a window came from.

Two things about that head decide whether any of it measures what it is meant to.

It is asked the question conditionally. Each capture belongs to exactly one class, so
naming the capture with nothing else to go on is naming the class in different words, and
a representation made independent of that would have lost the label rather than the
session. So the true class goes in as an input and the head's answer is confined to the
captures of that class. What it is asked is which of *this class's* sessions the window
came from, which is the question the feature-level work asked of the raw columns.

And it is only asked where the question exists. A per-window flag gates it: the reversed
representation is multiplied by that flag, so a window from a class recorded once, or from
a capture held back for probing, sends nothing back through the reversal at all.

The reversal strength is a variable rather than a constant, so a sweep assigns it and
reuses the same built model. It starts at zero here, where the model is exactly the
single-head model with an inert head attached.

In [ ]:
import keras
import tensorflow as tf

from src import invariance as ivar
from src import sequence as sq

keras.utils.set_random_seed(SEED)


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    return ", ".join(
        str(tf.config.experimental.get_device_details(d).get("device_name", d.name))
        for d in devices
    )


ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}
print(f"environment : {ENVIRONMENT}")
print()

# The captures the adversary is allowed to see: the multi-capture ones, less those held
# back. Every other window in the corpus reaches the attack head and stops there.
DOMAIN_CAPTURES = sorted(set(MULTI_CAPTURES) - set(HELD_OUT_CAPTURES))
CLASS_OF_CAPTURE = {g: CLASS_OF_GROUP[g] for g in DOMAIN_CAPTURES}

MEMBERSHIP = ivar.class_capture_membership(CLASSES, DOMAIN_CAPTURES, CLASS_OF_CAPTURE)
ADVERSARY_CHANCE = ivar.chance_rate(MEMBERSHIP)

LAMBDA = 0.0

def fresh_model(lam):
    """A model at its initial weights, seeded, every time.

    A function rather than a built object, because a model is stateful and a cell that
    fits one changes it. Building here and fitting in a later cell would mean the second
    time that cell ran it carried on from where the first left off, and a run meant to
    start from initialisation would silently be a warm start. Every fit calls this first.

    The encoder is built inside, for the same reason: it holds the weights the fit moves.
    """
    keras.utils.set_random_seed(SEED)
    return ivar.build(
        encoder=sq.record_encoder(len(FEATURES), len(CLASSES)),
        n_features=len(FEATURES),
        n_classes=len(CLASSES),
        n_captures=len(DOMAIN_CAPTURES),
        window=WINDOW,
        membership=MEMBERSHIP,
        lstm_units=sq.LSTM_UNITS,
        lam=lam,
    )


MODEL = fresh_model(LAMBDA)

MODEL.summary(line_length=110)
print()
print("inputs")
for name, tensor in MODEL.input.items():
    print(f"  {name:<14} {tuple(tensor.shape)}")
print("outputs")
for name, tensor in MODEL.output.items():
    print(f"  {name:<14} {tuple(tensor.shape)}")
print()

SPLIT = ivar.parameter_split(MODEL)
REFERENCE = sq.build_model(
    len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS
)
ENCODER_CHECK = sq.encoder_matches_baseline(MODEL, len(FEATURES), len(CLASSES))

print(f"trunk and attack head : {SPLIT['trunk_and_attack_head']:>8,}")
print(f"the adversary adds    : {SPLIT['adversary']:>8,}")
print(f"total                 : {SPLIT['total']:>8,}")
print(f"single-head model     : {REFERENCE.count_params():>8,}")
print()
print(f"captures the adversary sees : {len(DOMAIN_CAPTURES)} of {len(MULTI_CAPTURES)}")
print(f"reversal strength lambda    : {LAMBDA:g}")
print(f"guessing rate for its task  : {ADVERSARY_CHANCE:.4f}")

assert ENCODER_CHECK["agrees"], "the encoder in the trunk is not the published encoder"
assert SPLIT["trunk_and_attack_head"] == int(REFERENCE.count_params()), (
    "the trunk and attack head are not the same size as the single-head model, so the "
    "classifier being trained here is not the classifier it would be compared against"
)
assert len(DOMAIN_CAPTURES) == len(MULTI_CAPTURES) - len(HELD_OUT_CAPTURES)
assert MEMBERSHIP.shape == (len(CLASSES), len(DOMAIN_CAPTURES))
assert bool((MEMBERSHIP.sum(axis=0) == 1).all()), (
    "a capture reaches more than one class, or none"
)
assert abs(ADVERSARY_CHANCE - CHANCE_AFTER) < 1e-9, (
    "the adversary's chance rate disagrees with the one counted from the captures"
)

del REFERENCE

Training it, as a pilot. This is a short run at one seed to see whether the two heads
train together at all, not a result: the epochs are cut from the ten the rest of the
project uses, and nothing here is compared against anything.

The loss is the attack cross-entropy plus the capture cross-entropy. The second is summed
over the windows the adversary is allowed to see and divided by how many there were, so a
batch holding few of them does not contribute less per window than one holding many, and
the same flag that gates the reversed representation is the sample weight, so a window
outside that set is excluded twice: no gradient reaches the trunk through the adversary,
and no loss is counted for it.

The captures held back are kept out of the fit entirely, so what they can be asked later
is a question about sessions the model never saw. The switch below says so plainly, and
turning it on puts them back into the attack head's training data, which would leave them
useful for measuring what the adversary was denied and not for measuring generalisation to
an unseen session.

The reversal strength is set once at the top. Everything the run needs to be read later
goes into the config, and the fit, the scoring and the save are one statement each so a
dropped session cannot leave a trained model with nothing written about it.

In [ ]:
import time

from baselines import mohammadi as mo
from src import runs as rn

# The knobs for this run.
CAPTURE_LOSS = True
LAMBDA = 0.5
PILOT_EPOCHS = 2
TRAIN_ON_HELD_OUT = False

# With the capture loss off the objective is the attack cross-entropy alone, which is the
# single-head model trained and scored exactly as the sweep's runs are. That is the thing
# every point of the sweep has to be read against, and it is not the same statement as
# lambda at zero: lambda at zero leaves the adversary in the objective and multiplies by
# zero what it sends back, so its head still trains on a loss that is still reported.
CAPTURE_WEIGHT = 1.0 if CAPTURE_LOSS else 0.0
RUN_ID = "pilot_adversarial" if CAPTURE_LOSS else "pilot_attack_only"
OUT_DIR = ARTIFACTS / "NB11"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# What each head is asked for, per training window.
X_TRAIN = sq.reshape(TRAIN["X"])
CLASS_ONEHOT = np.eye(len(CLASSES), dtype="float32")[TRAIN["y"]]
ATTACK_TARGETS = CLASS_ONEHOT

CAPTURE_COLUMN = np.array(
    [{g: i for i, g in enumerate(DOMAIN_CAPTURES)}.get(GROUP_NAMES[c], -1) for c in GROUP_CODES]
)
# The adversary sees the earlier part of each capture it trains on, and not the part kept
# back to probe it with. Everything else the flag would have let through is still in the
# attack head's training data; it is only the adversary that is held off.
IN_DOMAIN = np.zeros((len(TRAIN["y"]), 1), dtype="float32")
IN_DOMAIN[ADVERSARY_INDEX] = 1.0
CAPTURE_TARGETS = np.zeros((len(TRAIN["y"]), len(DOMAIN_CAPTURES)), dtype="float32")
INSIDE = np.flatnonzero(CAPTURE_COLUMN >= 0)
CAPTURE_TARGETS[INSIDE, CAPTURE_COLUMN[INSIDE]] = 1.0
# A window the adversary may not see still needs a target the loss can evaluate, because
# an all-zero row is not a distribution. The weight throws the answer away.
CAPTURE_TARGETS[CAPTURE_COLUMN < 0, 0] = 1.0

FIT_ROWS = (np.arange(len(TRAIN["y"])) if TRAIN_ON_HELD_OUT
            else np.flatnonzero(~HELD_OUT_MASK))
EVAL_ROWS = np.flatnonzero(HELD_OUT_MASK)

print(f"run                  : {RUN_ID}")
print(f"objective            : "
      + ("attack cross-entropy alone; the adversary is built but contributes nothing"
         if not CAPTURE_LOSS else
         "attack cross-entropy plus the capture cross-entropy through the reversal"))
print(f"lambda               : {LAMBDA:g}"
      + ("   (inert: the capture loss is not in the objective)" if not CAPTURE_LOSS else ""))
print(f"epochs               : {PILOT_EPOCHS} (the full protocol is {int(CFG['training']['epochs'])})")
print(f"batch size           : {int(CFG['training']['batch_size'])}")
print(f"seed                 : {SEED}")
print(f"windows fitted on    : {len(FIT_ROWS):,} of {len(TRAIN['y']):,}")
print(f"  the adversary sees : {int(IN_DOMAIN[FIT_ROWS].sum()):,} "
      f"(the earlier part of each of the {len(DOMAIN_CAPTURES)} captures it trains on)")
print(f"  kept back to probe : {len(PROBE_INDEX):,} (the later part of those same captures)")
print(f"held-out windows     : {len(EVAL_ROWS):,}, "
      f"{'in the fit' if TRAIN_ON_HELD_OUT else 'kept out of the fit and scored at the end'}")
print()

assert int(IN_DOMAIN.sum()) == len(ADVERSARY_INDEX), (
    "the windows flagged for the adversary are not the adversary portion of the split"
)
assert int(IN_DOMAIN[PROBE_INDEX].sum()) == 0, (
    "a window kept back to probe with is flagged for the adversary to train on"
)
assert bool((CAPTURE_TARGETS.sum(axis=1) == 1).all()), "a capture target is not a distribution"
if not TRAIN_ON_HELD_OUT:
    assert int(IN_DOMAIN[EVAL_ROWS].sum()) == 0, "a held-out window is flagged for the adversary"

# Rebuilt at its initial weights before every fit. Without this, running this cell a
# second time to change a switch would carry on from the model the first run left behind,
# and the second run would report a warm start as though it were a fresh one.
MODEL = fresh_model(LAMBDA)
MODEL.compile(optimizer=mo.COMPILE["optimizer"])
print(f"model rebuilt at its initial weights, reversal strength "
      f"{float(MODEL.get_layer('gradient_reversal').lam.numpy()):g}")
print()

STARTED = time.perf_counter()
HISTORY = ivar.fit_adversarial(
    MODEL,
    X=X_TRAIN,
    attack_targets=ATTACK_TARGETS,
    class_onehot=CLASS_ONEHOT,
    in_domain=IN_DOMAIN,
    capture_targets=CAPTURE_TARGETS,
    epochs=PILOT_EPOCHS,
    batch_size=int(CFG["training"]["batch_size"]),
    seed=SEED,
    capture_weight=CAPTURE_WEIGHT,
    rows=FIT_ROWS,
    checkpoint_path=OUT_DIR / f"{RUN_ID}_checkpoint.keras",
    verbose=2,
)
TRAIN_SECONDS = time.perf_counter() - STARTED

# Scored on the captures held back, which is the one capture-disjoint evaluation this
# notebook has: eight classes, one session each, none of them seen by the fit.
STARTED = time.perf_counter()
EVAL_PROBABILITIES = MODEL.predict(
    {"windows": X_TRAIN[EVAL_ROWS], "class_onehot": CLASS_ONEHOT[EVAL_ROWS],
     "in_domain": np.zeros((len(EVAL_ROWS), 1), dtype="float32")},
    batch_size=512, verbose=0,
)["attack"]
INFERENCE_SECONDS = time.perf_counter() - STARTED

# Can the session still be named from what the trunk produces? Asked of windows the
# adversary never trained on, using the forest and the split the capture-identification
# measurement in this project has used from the start, read out of its own record rather
# than typed again here.
NB03_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB03_verdict.json",
     ARTIFACTS / "NB03" / "NB03_verdict.json"],
    "the settings the capture-identification measurement uses",
)
FOREST = json.loads(NB03_PATH.read_text())["forest"]

STARTED = time.perf_counter()
PROBE_DRAW, PROBE_PER_CAPTURE = ivar.equal_draw(PROBE_INDEX, GROUP_CODES, seed=SEED)
PROBE_H = ivar.representation(MODEL, X_TRAIN, PROBE_DRAW)
PROBE_ROWS = ivar.capture_probe(
    PROBE_H,
    [GROUP_NAMES[c] for c in GROUP_CODES[PROBE_DRAW]],
    [CLASSES[c] for c in TRAIN["y"][PROBE_DRAW]],
    n_estimators=FOREST["n_estimators"],
    min_samples_leaf=FOREST["min_samples_leaf"],
    test_fraction=FOREST["test_fraction"],
    seed=SEED,
)
PROBE = pd.DataFrame(PROBE_ROWS)
PROBE_AT = ivar.probe_summary(PROBE_ROWS)
PROBE_SECONDS = time.perf_counter() - STARTED

assert abs(PROBE_AT["class_held_fixed_chance"] - ADVERSARY_CHANCE) < 1e-9, (
    "the probe is not asking the question the adversary was trained against"
)

LABELS = np.asarray(CLASSES, dtype=str)
Y_TRUE = TRAIN["y"][EVAL_ROWS].astype("int8")
Y_PRED = EVAL_PROBABILITIES.argmax(axis=1).astype("int8")

METRICS = rn.classification_metrics(
    LABELS[Y_TRUE.astype("int64")], LABELS[Y_PRED.astype("int64")], labels=CLASSES
)

# The held-out captures cover the eight classes recorded more than once and nothing else.
# Scored against all nineteen labels, the eleven with no window here each take an F1 of
# zero and each count in the denominator, so the macro figure is a sum over eight divided
# by nineteen. Both are recorded: the first is what every other run in this project means
# by macro F1, and the second is what this evaluation actually measured.
PRESENT_CLASSES = [c for c in CLASSES if METRICS["support"][c] > 0]
ABSENT_CLASSES = [c for c in CLASSES if METRICS["support"][c] == 0]
MACRO_PRESENT = float(np.mean([METRICS["per_class_f1"][c] for c in PRESENT_CLASSES]))

METRICS["macro_f1_over_present_classes"] = MACRO_PRESENT
METRICS["chance_rate_over_all_labels"] = float(METRICS["chance_rate"])
# Corrected. This evaluation is an eight-class problem and the rate stored by default
# describes a nineteen-class one, which would overstate the margin accuracy clears.
METRICS["chance_rate"] = 1.0 / max(len(PRESENT_CLASSES), 1)
METRICS["evaluation"] = {
    "labels_scored_against": len(CLASSES),
    "classes_present": PRESENT_CLASSES,
    "classes_absent": ABSENT_CLASSES,
    "macro_f1_over_all_labels": float(METRICS["macro_f1"]),
    "macro_f1_over_present_classes": MACRO_PRESENT,
    "chance_over_all_labels": float(METRICS["chance_rate_over_all_labels"]),
    "chance_over_present_classes": float(METRICS["chance_rate"]),
    "reading": (
        f"macro_f1 is over all {len(CLASSES)} labels, as everywhere else in this project, "
        f"and the {len(ABSENT_CLASSES)} classes with no window here each contribute an F1 "
        f"of 0 to it. macro_f1_over_present_classes is the same sum over the "
        f"{len(PRESENT_CLASSES)} classes that were scored. The two differ by a factor of "
        f"{len(CLASSES)}/{len(PRESENT_CLASSES)}. chance_rate has been corrected to the "
        f"{len(PRESENT_CLASSES)}-class rate this evaluation runs at; "
        "chance_rate_over_all_labels keeps the figure it replaced"
    ),
    "not_comparable_to": (
        "the 19-class figures recorded for other runs, which are scored on the full test "
        "partition rather than on held-out captures of eight classes"
    ),
}

assert set(PRESENT_CLASSES) == set(MULTI_CLASSES), (
    "the held-out captures do not cover exactly the classes recorded more than once, so "
    "the evaluation is not the one the holdout was built for"
)

METRICS["train_seconds"] = round(TRAIN_SECONDS, 3)
METRICS["inference_seconds"] = round(INFERENCE_SECONDS, 3)
METRICS["n_train"] = int(len(FIT_ROWS))
METRICS["n_parameters"] = int(MODEL.count_params())
METRICS["history"] = HISTORY
METRICS["parameters_by_head"] = SPLIT
METRICS["adversary"] = {
    "in_the_objective": CAPTURE_LOSS,
    "capture_loss_weight": CAPTURE_WEIGHT,
    "role": ("the comparator: the objective is the attack loss alone, so this is the "
             "single-head model trained and scored the same way as the sweep's runs"
             if not CAPTURE_LOSS else
             "the intervention: the trunk is pushed away from what names the session"),
    "lambda": LAMBDA,
    "captures": DOMAIN_CAPTURES,
    "n_captures": len(DOMAIN_CAPTURES),
    "chance_rate": ADVERSARY_CHANCE,
    "conditioned_on": "the true class, as an input, with the answer masked to that "
                      "class's own captures",
    "gated_by": "a per-window flag, applied both to the reversed representation and as "
                "the sample weight on the capture loss",
}
METRICS["capture_probe"] = {
    "question": "which of this class's captures did this window come from",
    "measured_on": "windows held back from the adversary inside the captures it trained on, "
                   "so the trunk was fitted on them and the adversary never optimised "
                   "against them",
    "windows": int(len(PROBE_DRAW)),
    "per_capture": int(PROBE_PER_CAPTURE),
    "captures": len(DOMAIN_CAPTURES),
    "forest": FOREST,
    "forest_from": str(NB03_PATH),
    "probe_seconds": round(PROBE_SECONDS, 3),
    "by_scope": PROBE_ROWS,
    **PROBE_AT,
    "reading": (
        "the class-held-fixed row is the one that matters. Pooled over every capture, "
        "naming the capture well could be naming the class, since a capture belongs to "
        "one class; with the class fixed the captures being told apart all carry the "
        "same label"
    ),
}

METRICS["scored_on"] = (
    f"the {len(EVAL_ROWS):,} windows of the {len(HELD_OUT_CAPTURES)} held-out captures, "
    f"{'which were also in the fit' if TRAIN_ON_HELD_OUT else 'none of which were in the fit'}"
)

CONFIG = {
    "run_id": RUN_ID,
    "parent": None,
    "notes": "pilot. Reduced epochs, one seed, no comparison drawn and no hypothesis "
             "tested. A root rather than a child, because the model, the inputs and the "
             "objective all differ from any run already recorded and no single change "
             "relates them",
    "model": "cnn_lstm_two_head: the published record encoder per record, one LSTM across "
             "the window, a softmax over the classes, and a capture head behind a gradient "
             "reversal",
    "task": "19-class",
    "split": "two_tier",
    "n_features": len(FEATURES),
    "window": WINDOW,
    "stride": STRIDE,
    "lstm_units": sq.LSTM_UNITS,
    "capture_loss": CAPTURE_LOSS,
    "capture_loss_weight": CAPTURE_WEIGHT,
    "lambda": LAMBDA,
    "epochs": PILOT_EPOCHS,
    "batch_size": int(CFG["training"]["batch_size"]),
    "seed": SEED,
    "train_on_held_out": TRAIN_ON_HELD_OUT,
    "held_out_captures": HELD_OUT_CAPTURES,
    "domain_captures": DOMAIN_CAPTURES,
    "observed": {
        "environment": ENVIRONMENT,
        "run_date": RUN_DATE,
        "git_sha": GIT_SHA,
        "git_dirty": GIT_DIRTY,
        "mode": "pilot",
    },
}

print()
print(f"scored on {len(EVAL_ROWS):,} windows of {len(HELD_OUT_CAPTURES)} held-out captures, "
      f"covering {len(PRESENT_CLASSES)} of the {len(CLASSES)} classes")
print(f"  {len(ABSENT_CLASSES)} classes have no window here and score F1 0 in any figure "
      "averaged over all of them")
print()
print(f"attack macro F1, over all {len(CLASSES)} labels          : {METRICS['macro_f1']:.4f}"
      f"   (comparable in kind to other runs, but not in what it was scored on)")
print(f"attack macro F1, over the {len(PRESENT_CLASSES)} classes present : {MACRO_PRESENT:.4f}"
      f"   (what this evaluation measured)")
print(f"attack accuracy                             : {METRICS['accuracy']:.4f}")
print(f"attack weighted F1                          : {METRICS['weighted_f1']:.4f}"
      "   (unaffected: an absent class carries no weight)")
print()
print(f"capture identification off the representation, on windows the adversary never saw")
print(PROBE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"  pooled over {PROBE_AT['n_classes'] and len(DOMAIN_CAPTURES)} captures"
      f"                : {PROBE_AT['pooled_accuracy']:.4f}"
      f"   against chance {PROBE_AT['pooled_chance']:.4f}")
print(f"  with the attack class held fixed     : "
      f"{PROBE_AT['class_held_fixed_accuracy']:.4f}"
      f"   against chance {PROBE_AT['class_held_fixed_chance']:.4f}")
print(f"  {PROBE_PER_CAPTURE:,} windows drawn from each of {len(DOMAIN_CAPTURES)} captures, "
      f"{len(PROBE_DRAW):,} in all")
print()
print(f"chance for the {len(PRESENT_CLASSES)}-class problem scored     : "
      f"{METRICS['chance_rate']:.4f}   (corrected; stored as "
      f"{METRICS['chance_rate_over_all_labels']:.4f} by default)")
print(f"always answering the largest class          : {METRICS['majority_class_rate']:.4f}")
print()
print(f"trained in {TRAIN_SECONDS / 60:.1f} minutes, scored in {INFERENCE_SECONDS:.1f} seconds")
if not CAPTURE_LOSS:
    print()
    print("This run is the comparator. The adversary was built and its loss reported, and "
          "none of")
    print("it reached the objective, so the trunk saw the attack loss and nothing else.")

RUN = rn.save_run_codes(
    OUT_DIR, RUN_ID, config=CONFIG, metrics=METRICS,
    y_true=Y_TRUE, y_pred=Y_PRED, labels=CLASSES, model=MODEL,
)